# Validação do Censo Escolar com Pandera

Validação do arquivo consolidado `trabalho/censo-escolar.csv`.

O schema valida 42 colunas harmonizadas entre 1995 e 2025, cobrindo tipos, domínios e regras de consistência dependentes de `ano_censo`.

## 0) Configuração

Importa bibliotecas, define caminhos do CSV e prepara a pasta dos relatórios de validação.

In [1]:
import os, re, warnings
from pathlib import Path

os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'
warnings.filterwarnings('ignore')

import pandas as pd
import pandera.pandas as pa
import pandera.errors as pe
from pandera import Check, Column

csv_candidates = [
  Path('censo-escolar.csv'),
  Path('trabalho/censo-escolar.csv'),
]
CSV_PATH = next((path for path in csv_candidates if path.exists()), csv_candidates[-1])
REPORT_DIR = CSV_PATH.parent / 'schema_reports'
BATCH_SIZE = 200_000
SAMPLE_SIZE = 500_000
MAX_EXEMPLOS_POR_CHECK = 2_000

REPORT_DIR.mkdir(parents=True, exist_ok=True)
for path in REPORT_DIR.glob('*.csv'):
  path.unlink()

if not CSV_PATH.exists():
  caminhos = ', '.join(str(path) for path in csv_candidates)
  raise FileNotFoundError(f'CSV consolidado não encontrado. Caminhos testados: {caminhos}')

print(f'Fonte configurada: {CSV_PATH}')
print(f'Relatórios: {REPORT_DIR}')

Fonte configurada: censo-escolar.csv
Relatórios: schema_reports


## 1) Estrutura do CSV

Conta linhas, lista as 42 colunas e confirma o tamanho do arquivo consolidado.

In [2]:
colunas_csv = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
total_linhas_csv = 0
for chunk in pd.read_csv(CSV_PATH, usecols=['ano_censo'], chunksize=BATCH_SIZE):
  total_linhas_csv += len(chunk)

print('=' * 70)
print('Metadados do arquivo CSV'.center(70))
print('=' * 70)
print(f'Linhas totais:      {total_linhas_csv:>15,}')
print(f'Colunas totais:     {len(colunas_csv):>15}')
print(f'Tamanho arquivo:    {CSV_PATH.stat().st_size / 1024**2:>14.1f} MB')

print()
print('Schema do arquivo consolidado')
print(f'{"Ordem":<8} {"Coluna":<32}')
print(f'{"-"*8} {"-"*32}')
for posicao, coluna in enumerate(colunas_csv, start=1):
  print(f'{posicao:<8} {coluna:<32}')

                       Metadados do arquivo CSV                       
Linhas totais:            7,376,443
Colunas totais:                  42
Tamanho arquivo:            1074.8 MB

Schema do arquivo consolidado
Ordem    Coluna                          
-------- --------------------------------
1        ano_censo                       
2        id_escola                       
3        co_municipio                    
4        no_municipio                    
5        sg_uf                           
6        co_uf                           
7        no_uf                           
8        no_regiao                       
9        co_regiao                       
10       no_entidade                     
11       dependencia_administrativa      
12       localizacao                     
13       situacao_funcionamento          
14       tp_localizacao_diferenciada     
15       tp_categoria_escola_privada     
16       in_regular                      
17       in_creche              

## 2) Amostra Inicial

Carrega uma amostra controlada para observar preenchimento, valores frequentes e colunas mais nulas.

In [3]:
print(f'Carregando amostra de {SAMPLE_SIZE:,} linhas para análise...')

batches, lidas = [], 0
for batch in pd.read_csv(CSV_PATH, chunksize=100_000, dtype='string'):
  batches.append(batch)
  lidas += len(batch)
  print(f'Lidas {lidas:,} linhas...', end='\r')
  if lidas >= SAMPLE_SIZE:
    break

amostra = pd.concat(batches, ignore_index=True).head(SAMPLE_SIZE)
print(f'\nAmostra carregada: {len(amostra):,} linhas')
print(f'Memória: {amostra.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB')

print('Informações da amostra:')
display(amostra.describe(include='all').T)

nulos_amostra = amostra.isna().mean().sort_values(ascending=False).rename('pct_nulos').reset_index()
nulos_amostra = nulos_amostra.rename(columns={'index': 'coluna'})
display(nulos_amostra.head(20))

Carregando amostra de 500,000 linhas para análise...
Lidas 500,000 linhas...
Amostra carregada: 500,000 linhas
Memória: 196.19 MB
Informações da amostra:


,count,unique,top,freq
ano_censo,500000,2,1996,256363
id_escola,500000,500000,0000000027,1
co_municipio,500000,12561,351506150308,2994
no_municipio,500000,4743,SAO PAULO,5753
sg_uf,500000,27,BA,66474
co_uf,0,0,NaN,NaN
no_uf,500000,27,Bahia,66474
no_regiao,0,0,NaN,NaN
co_regiao,0,0,NaN,NaN
no_entidade,0,0,NaN,NaN


,coluna,pct_nulos
0,no_entidade,1.0
1,co_uf,1.0
2,co_regiao,1.0
3,no_regiao,1.0
4,in_educacao_indigena,1.0
5,in_especial,1.0
6,in_predio_compartilhado,1.0
7,in_creche,1.0
8,tp_categoria_escola_privada,1.0
9,tp_localizacao_diferenciada,1.0


## 3) Contrato de Colunas

Compara as colunas do CSV com a lista esperada antes de construir o schema.

In [4]:
COLUNAS_ESPERADAS = [
  'ano_censo', 'id_escola', 'co_municipio', 'no_municipio', 'sg_uf', 'co_uf', 'no_uf',
  'no_regiao', 'co_regiao', 'no_entidade', 'dependencia_administrativa', 'localizacao',
  'situacao_funcionamento', 'tp_localizacao_diferenciada', 'tp_categoria_escola_privada',
  'in_regular', 'in_creche', 'in_pre_escola', 'in_fundamental', 'in_medio', 'in_eja',
  'in_profissionalizante', 'in_especial', 'in_educacao_indigena', 'in_predio_escolar',
  'in_predio_compartilhado', 'in_agua_potavel', 'in_agua_rede_publica',
  'in_agua_poco_artesiano', 'in_agua_inexistente', 'in_energia_rede_publica',
  'in_energia_gerador', 'in_energia_inexistente', 'qt_mat_bas', 'qt_mat_fund',
  'qt_mat_med', 'qt_doc_bas', 'qt_doc_fund', 'qt_doc_med', 'qt_tur_bas',
  'qt_tur_fund', 'qt_tur_med',
]

faltantes = [c for c in COLUNAS_ESPERADAS if c not in colunas_csv]
extras = [c for c in colunas_csv if c not in COLUNAS_ESPERADAS]

print(f'Colunas esperadas: {len(COLUNAS_ESPERADAS)}')
print(f'Colunas no CSV: {len(colunas_csv)}')
print(f'Colunas ausentes: {faltantes}')
print(f'Colunas extras: {extras}')

if faltantes or extras:
  raise ValueError('O CSV não corresponde ao contrato de 42 colunas esperadas.')

Colunas esperadas: 42
Colunas no CSV: 42
Colunas ausentes: []
Colunas extras: []


## 4) Schema Pandera

Define tipos, domínios e regras RN-01 a RN-16 que serão aplicadas ao CSV.

In [5]:
CODIGOS_UF = [11, 12, 13, 14, 15, 16, 17, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 35, 41, 42, 43, 50, 51, 52, 53]
CODIGOS_REGIAO = [1, 2, 3, 4, 5]
DEPENDENCIAS = ['Federal', 'Estadual', 'Municipal', 'Privada']
LOCALIZACOES = ['Urbana', 'Rural']
SITUACOES = ['Ativa', 'Paralisada', 'Paralisado', 'Extinta', 'Extinto']

COLUNAS_IN = [c for c in COLUNAS_ESPERADAS if c.startswith('in_')]
COLUNAS_QT = [c for c in COLUNAS_ESPERADAS if c.startswith('qt_')]

def coluna_int(checks=None, nullable=True, description=None):
  return Column('Int64', nullable=nullable, checks=checks or [], coerce=True, description=description)

def coluna_str(checks=None, nullable=True, description=None):
  return Column('string', nullable=nullable, checks=checks or [], coerce=True, description=description)

schema_cols = {
  'ano_censo': coluna_int([Check.in_range(1995, 2025)], nullable=False, description='Ano do Censo Escolar'),
  'id_escola': coluna_str([Check.str_matches(r'^\d+$')], nullable=False, description='Código da escola'),
  'co_municipio': coluna_str([Check.str_matches(r'^\d+$')], description='Código do município'),
  'no_municipio': coluna_str([Check.str_length(min_value=1)], description='Nome do município'),
  'sg_uf': coluna_str([Check.str_matches(r'^[A-Z]{2}$')], description='Sigla da UF'),
  'co_uf': coluna_int([Check.isin(CODIGOS_UF)], description='Código da UF'),
  'no_uf': coluna_str([Check.str_length(min_value=1)], description='Nome da UF'),
  'no_regiao': coluna_str([Check.str_length(min_value=1)], description='Nome da região'),
  'co_regiao': coluna_int([Check.isin(CODIGOS_REGIAO)], description='Código da região'),
  'no_entidade': coluna_str([Check.str_length(min_value=1)], description='Nome da escola'),
  'dependencia_administrativa': coluna_str([Check.isin(DEPENDENCIAS)], description='Dependência administrativa'),
  'localizacao': coluna_str([Check.isin(LOCALIZACOES)], description='Localização'),
  'situacao_funcionamento': coluna_str([Check.isin(SITUACOES)], description='Situação de funcionamento'),
  'tp_localizacao_diferenciada': coluna_int([Check.ge(0)], description='Tipo de localização diferenciada'),
  'tp_categoria_escola_privada': coluna_int([Check.isin([0, 1, 2, 3, 4])], description='Categoria de escola privada'),
}

for col in COLUNAS_IN:
  schema_cols[col] = coluna_int([Check.isin([0, 1])], description='Indicador binário')

for col in COLUNAS_QT:
  schema_cols[col] = coluna_int([Check.ge(0)], description='Contagem não negativa')

schema_cols = {col: schema_cols[col] for col in COLUNAS_ESPERADAS}

def id_escola_valido(df):
  tamanho = df['id_escola'].astype('string').str.len()
  return df['id_escola'].isna() | ((df['ano_censo'] == 1995) & tamanho.eq(10)) | ((df['ano_censo'] >= 1996) & tamanho.eq(8))

def municipio_valido(df):
  tamanho = df['co_municipio'].astype('string').str.len()
  return df['co_municipio'].isna() | ((df['ano_censo'] == 1995) & tamanho.eq(14)) | (df['ano_censo'].between(1996, 2006) & tamanho.eq(12)) | ((df['ano_censo'] >= 2007) & tamanho.eq(7))

def localizacao_diferenciada_valida(df):
  ano = df['ano_censo']
  col = df['tp_localizacao_diferenciada']
  return (
    col.isna()
    | (ano.between(1995, 2011) & col.isin([0, 1, 2, 3]))
    | (ano.between(2012, 2018) & col.isin([0, 1, 2, 3, 4, 5, 6]))
    | (ano.between(2019, 2022) & col.isin([0, 1, 2, 3]))
    | (ano.between(2023, 2025) & col.isin([0, 1, 2, 3, 8]))
  )

def total_maior_igual_parte(df, total, parte):
  return df[[total, parte]].isna().any(axis=1) | (df[total] >= df[parte])

def indicador_compativel_com_quantidade(df, indicador, quantidade):
  return df[[indicador, quantidade]].isna().any(axis=1) | (df[quantidade] == 0) | (df[indicador] == 1)

schema_censo = pa.DataFrameSchema(
  schema_cols,
  checks=[
    Check(lambda df: ~df.duplicated(subset=['ano_censo', 'id_escola']).any(), error='RN-01: chave (ano_censo, id_escola) não pode duplicar'),
    Check(id_escola_valido, error='RN-02: id_escola deve ter tamanho compatível com o ano'),
    Check(municipio_valido, error='RN-03: co_municipio deve ter tamanho compatível com o ano'),
    Check(localizacao_diferenciada_valida, error='RN-04: tp_localizacao_diferenciada fora do domínio válido para o ano'),
    Check(lambda df: (df['ano_censo'] < 2007) | df[['situacao_funcionamento', 'qt_mat_bas']].isna().any(axis=1) | (df['situacao_funcionamento'] != 'Ativa') | (df['qt_mat_bas'] > 0), error='RN-05: escola ativa deve ter qt_mat_bas > 0 quando disponível'),
    Check(lambda df: df[['in_agua_inexistente', 'in_agua_rede_publica']].isna().any(axis=1) | (df['in_agua_inexistente'] == 0) | (df['in_agua_rede_publica'] == 0), error='RN-06: água inexistente não deve coexistir com rede pública'),
    Check(lambda df: df[['in_energia_inexistente', 'in_energia_rede_publica']].isna().any(axis=1) | (df['in_energia_inexistente'] == 0) | (df['in_energia_rede_publica'] == 0), error='RN-07: energia inexistente não deve coexistir com rede pública'),
    Check(lambda df: (df['ano_censo'] < 2007) | df[['qt_mat_bas', 'qt_mat_fund', 'qt_mat_med']].isna().any(axis=1) | (df['qt_mat_bas'] >= df['qt_mat_fund'] + df['qt_mat_med']), error='RN-08: qt_mat_bas deve ser >= qt_mat_fund + qt_mat_med quando disponível'),
    Check(lambda df: total_maior_igual_parte(df, 'qt_doc_bas', 'qt_doc_fund'), error='RN-09: qt_doc_fund não pode ser maior que qt_doc_bas'),
    Check(lambda df: total_maior_igual_parte(df, 'qt_doc_bas', 'qt_doc_med'), error='RN-10: qt_doc_med não pode ser maior que qt_doc_bas'),
    Check(lambda df: total_maior_igual_parte(df, 'qt_tur_bas', 'qt_tur_fund'), error='RN-11: qt_tur_fund não pode ser maior que qt_tur_bas'),
    Check(lambda df: total_maior_igual_parte(df, 'qt_tur_bas', 'qt_tur_med'), error='RN-12: qt_tur_med não pode ser maior que qt_tur_bas'),
    Check(lambda df: indicador_compativel_com_quantidade(df, 'in_fundamental', 'qt_mat_fund'), error='RN-13: qt_mat_fund > 0 exige in_fundamental = 1'),
    Check(lambda df: indicador_compativel_com_quantidade(df, 'in_medio', 'qt_mat_med'), error='RN-14: qt_mat_med > 0 exige in_medio = 1'),
    Check(lambda df: df[['qt_mat_fund', 'qt_tur_fund']].isna().any(axis=1) | (df['qt_tur_fund'] == 0) | (df['qt_mat_fund'] > 0), error='RN-15: qt_tur_fund > 0 exige qt_mat_fund > 0'),
    Check(lambda df: df[['qt_mat_med', 'qt_tur_med']].isna().any(axis=1) | (df['qt_tur_med'] == 0) | (df['qt_mat_med'] > 0), error='RN-16: qt_tur_med > 0 exige qt_mat_med > 0'),
  ],
  coerce=True,
  strict=True,
  name='schema_censo_escolar_harmonizado',
)

print(f'Schema criado com {len(schema_censo.columns)} colunas e {len(schema_censo.checks)} regras de dataframe.')

Schema criado com 42 colunas e 16 regras de dataframe.


## 5) Validação em Batches

Valida o arquivo em partes, resume as falhas e classifica cada regra por tipo e severidade.

In [6]:
REGRAS = {
  'RN-01': ('erro_estrutural', 'alta', True, 'duplicidade de chave compromete a unidade observacional'),
  'RN-02': ('erro_estrutural', 'alta', True, 'formato de id_escola incompatível com o ano'),
  'RN-03': ('erro_estrutural', 'alta', True, 'formato de co_municipio incompatível com o ano'),
  'RN-04': ('erro_de_dominio', 'média', False, 'código fora do domínio histórico documentado'),
  'RN-05': ('alerta_de_qualidade', 'média', False, 'escola ativa sem matrícula básica positiva; requer investigação'),
  'RN-06': ('erro_de_consistencia', 'média', False, 'água inexistente coexistindo com rede pública'),
  'RN-07': ('erro_de_consistencia', 'média', False, 'energia inexistente coexistindo com rede pública'),
  'RN-08': ('alerta_de_qualidade', 'média', False, 'matrícula básica menor que a soma de fundamental e médio'),
  'RN-09': ('erro_de_consistencia', 'média', False, 'docentes no fundamental acima do total da educação básica'),
  'RN-10': ('erro_de_consistencia', 'média', False, 'docentes no médio acima do total da educação básica'),
  'RN-11': ('erro_de_consistencia', 'média', False, 'turmas no fundamental acima do total da educação básica'),
  'RN-12': ('erro_de_consistencia', 'média', False, 'turmas no médio acima do total da educação básica'),
  'RN-13': ('erro_de_consistencia', 'média', False, 'matrícula no fundamental sem indicador de oferta correspondente'),
  'RN-14': ('erro_de_consistencia', 'média', False, 'matrícula no médio sem indicador de oferta correspondente'),
  'RN-15': ('alerta_de_qualidade', 'média', False, 'turma no fundamental sem matrícula correspondente'),
  'RN-16': ('alerta_de_qualidade', 'média', False, 'turma no médio sem matrícula correspondente'),
}

def classificar_check(check):
  regra = re.search(r'RN-\d+', str(check))
  if regra:
    codigo = regra.group(0)
    tipo, severidade, fatal, interpretacao = REGRAS.get(codigo, ('alerta_de_qualidade', 'baixa', False, 'regra não classificada'))
    return pd.Series({'regra': codigo, 'tipo_falha': tipo, 'severidade': severidade, 'fatal': fatal, 'interpretacao': interpretacao})
  return pd.Series({'regra': 'COLUNA', 'tipo_falha': 'erro_de_dominio', 'severidade': 'média', 'fatal': False, 'interpretacao': 'falha de tipo, domínio ou nulidade em coluna'})

resumos = []
exemplos_por_check = {}
linhas_validadas = 0

for n_batch, df_batch in enumerate(pd.read_csv(CSV_PATH, usecols=COLUNAS_ESPERADAS, chunksize=BATCH_SIZE, dtype='string'), start=1):
  linhas_validadas += len(df_batch)

  try:
    schema_censo.validate(df_batch, lazy=True)
  except pe.SchemaErrors as err:
    falhas = err.failure_cases.copy()
    falhas['lote'] = n_batch

    resumos.append(
      falhas.groupby(['schema_context', 'column', 'check'], dropna=False)
      .size()
      .reset_index(name='qtd_falhas_pandera')
    )

    for check, grupo in falhas.groupby('check', dropna=False):
      chave = str(check)
      salvos = sum(len(parte) for parte in exemplos_por_check.get(chave, []))
      restante = MAX_EXEMPLOS_POR_CHECK - salvos
      if restante > 0:
        exemplos_por_check.setdefault(chave, []).append(grupo.head(restante))

  if n_batch % 5 == 0:
    print(f'Batch {n_batch}: {linhas_validadas:,} linhas validadas')

if resumos:
  resumo_falhas = (
    pd.concat(resumos, ignore_index=True)
    .groupby(['schema_context', 'column', 'check'], dropna=False)['qtd_falhas_pandera']
    .sum()
    .reset_index()
    .sort_values('qtd_falhas_pandera', ascending=False)
  )
  resumo_falhas = pd.concat([resumo_falhas, resumo_falhas['check'].apply(classificar_check)], axis=1)
  resumo_falhas['qtd_linhas_estimadas'] = resumo_falhas['qtd_falhas_pandera']
else:
  resumo_falhas = pd.DataFrame(columns=['schema_context', 'column', 'check', 'qtd_falhas_pandera', 'regra', 'tipo_falha', 'severidade', 'fatal', 'interpretacao', 'qtd_linhas_estimadas'])

partes_exemplos = [parte for partes in exemplos_por_check.values() for parte in partes]
amostra_falhas = pd.concat(partes_exemplos, ignore_index=True) if partes_exemplos else pd.DataFrame()
if not amostra_falhas.empty:
  amostra_falhas = pd.concat([amostra_falhas, amostra_falhas['check'].apply(classificar_check)], axis=1)

visao_geral = (
  resumo_falhas.groupby(['regra', 'tipo_falha', 'severidade', 'fatal', 'interpretacao'], dropna=False)
  .agg(qtd_falhas_pandera=('qtd_falhas_pandera', 'sum'), qtd_linhas_estimadas=('qtd_linhas_estimadas', 'max'))
  .reset_index()
  .sort_values('qtd_falhas_pandera', ascending=False)
) if not resumo_falhas.empty else pd.DataFrame(columns=['regra', 'tipo_falha', 'severidade', 'fatal', 'interpretacao', 'qtd_falhas_pandera', 'qtd_linhas_estimadas'])

print(f'Linhas validadas: {linhas_validadas:,}')
print(f'Falhas registradas pelo Pandera: {resumo_falhas["qtd_falhas_pandera"].sum() if not resumo_falhas.empty else 0:,}')
print(f'Linhas estimadas com alerta/erro: {visao_geral["qtd_linhas_estimadas"].sum() if not visao_geral.empty else 0:,}')
display(visao_geral)
display(resumo_falhas.head(30))

Batch 5: 1,000,000 linhas validadas
Batch 10: 2,000,000 linhas validadas
Batch 15: 3,000,000 linhas validadas
Batch 20: 4,000,000 linhas validadas
Batch 25: 5,000,000 linhas validadas
Batch 30: 6,000,000 linhas validadas
Batch 35: 7,000,000 linhas validadas
Linhas validadas: 7,376,443
Falhas registradas pelo Pandera: 1,744,395
Linhas estimadas com alerta/erro: 41,556


,regra,tipo_falha,severidade,fatal,interpretacao,qtd_falhas_pandera,qtd_linhas_estimadas
0,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,1619268,38554
1,RN-13,erro_de_consistencia,média,False,matrícula no fundamental sem indicador de ofer...,65918,1575
2,RN-14,erro_de_consistencia,média,False,matrícula no médio sem indicador de oferta cor...,47286,1142
3,RN-16,alerta_de_qualidade,média,False,turma no médio sem matrícula correspondente,11923,285


,schema_context,column,check,qtd_falhas_pandera,regra,tipo_falha,severidade,fatal,interpretacao,qtd_linhas_estimadas
0,DataFrameSchema,ano_censo,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
4,DataFrameSchema,co_municipio,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
36,DataFrameSchema,in_agua_rede_publica,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
32,DataFrameSchema,in_agua_potavel,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
16,DataFrameSchema,dependencia_administrativa,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
20,DataFrameSchema,id_escola,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
12,DataFrameSchema,co_uf,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
8,DataFrameSchema,co_regiao,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
28,DataFrameSchema,in_agua_poco_artesiano,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554
24,DataFrameSchema,in_agua_inexistente,RN-05: escola ativa deve ter qt_mat_bas > 0 qu...,38554,RN-05,alerta_de_qualidade,média,False,escola ativa sem matrícula básica positiva; re...,38554


## 6) Exportação dos Relatórios

Gera os CSVs finais com contrato do schema, resumo da validação e amostra de falhas.

In [7]:
contrato_schema = pd.DataFrame({
  'coluna': list(schema_censo.columns.keys()),
  'tipo': [str(col.dtype) for col in schema_censo.columns.values()],
  'permite_nulo': [col.nullable for col in schema_censo.columns.values()],
  'descricao': [col.description for col in schema_censo.columns.values()],
  'checagens': ['; '.join(str(check) for check in col.checks) for col in schema_censo.columns.values()],
})

resumo_export = resumo_falhas.rename(columns={
  'schema_context': 'contexto_schema',
  'column': 'coluna',
  'check': 'checagem',
})

amostra_export = amostra_falhas.rename(columns={
  'schema_context': 'contexto_schema',
  'column': 'coluna',
  'check': 'checagem',
  'check_number': 'numero_checagem',
  'failure_case': 'valor_falha',
  'index': 'indice_linha',
})

contrato_schema.to_csv(REPORT_DIR / 'contrato_schema.csv', index=False)
resumo_export.to_csv(REPORT_DIR / 'resumo_validacao.csv', index=False)
amostra_export.to_csv(REPORT_DIR / 'amostra_falhas.csv', index=False)

print('Arquivos gerados:')
for path in sorted(REPORT_DIR.glob('*.csv')):
  print('-', path)

display(contrato_schema)

Arquivos gerados:
- schema_reports/amostra_falhas.csv
- schema_reports/contrato_schema.csv
- schema_reports/resumo_validacao.csv


,coluna,tipo,permite_nulo,descricao,checagens
0,ano_censo,Int64,False,Ano do Censo Escolar,"<Check in_range: in_range(1995, 2025)>"
1,id_escola,string[pyarrow],False,Código da escola,<Check str_matches: str_matches('^\d+$')>
2,co_municipio,string[pyarrow],True,Código do município,<Check str_matches: str_matches('^\d+$')>
3,no_municipio,string[pyarrow],True,Nome do município,"<Check str_length: str_length(1, None)>"
4,sg_uf,string[pyarrow],True,Sigla da UF,<Check str_matches: str_matches('^[A-Z]{2}$')>
5,co_uf,Int64,True,Código da UF,"<Check isin: isin([11, 12, 13, 14, 15, 16, 17,..."
6,no_uf,string[pyarrow],True,Nome da UF,"<Check str_length: str_length(1, None)>"
7,no_regiao,string[pyarrow],True,Nome da região,"<Check str_length: str_length(1, None)>"
8,co_regiao,Int64,True,Código da região,"<Check isin: isin([1, 2, 3, 4, 5])>"
9,no_entidade,string[pyarrow],True,Nome da escola,"<Check str_length: str_length(1, None)>"
